# RefCOCOg dataset sanity check

This notebook validates that referring expressions, COCO images, and ground-truth boxes load correctly. It is intentionally not the training or GRPO notebook.

In [ ]:
import os
from pathlib import Path

BASE_DIR = Path(os.getenv("VLM_BASE_DIR", "/scratch/ambpdc/vlm"))
COCO_DIR = Path(os.getenv("COCO_DIR", str(BASE_DIR / "coco")))
CACHE_DIR = BASE_DIR / "cache"
OUTPUT_DIR = BASE_DIR / "outputs" / "dataset_sanity"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set these before importing datasets so downloads use HPC storage.
os.environ["HF_HOME"] = str(CACHE_DIR / "huggingface")
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR / "huggingface" / "hub")
os.environ["HF_DATASETS_CACHE"] = str(CACHE_DIR / "huggingface" / "datasets")

# Keep remote image downloads disabled on HPC. For a temporary Colab-only
# check, set ALLOW_REMOTE_IMAGE_DOWNLOAD=1 before running this notebook.
ALLOW_REMOTE_IMAGE_DOWNLOAD = os.getenv("ALLOW_REMOTE_IMAGE_DOWNLOAD", "0") == "1"
COCO_VERIFY_SSL = os.getenv("COCO_VERIFY_SSL", "1") == "1"

print("COCO directory:", COCO_DIR)
print("Output directory:", OUTPUT_DIR)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import requests
from datasets import load_dataset
from PIL import Image

ds = load_dataset("jxu124/refcocog", split="train")

SMOKE_TEST = os.getenv("SMOKE_TEST", "1") == "1"
if SMOKE_TEST:
    ds = ds.select(range(min(50, len(ds))))

print(ds)
print(ds.column_names)

In [ ]:
def find_local_coco_image(file_name):
    for split_dir in ("train2014", "val2014"):
        candidate = COCO_DIR / split_dir / file_name
        if candidate.exists():
            return candidate
    return None


def load_coco_image(file_name):
    local_path = find_local_coco_image(file_name)
    if local_path is not None:
        return Image.open(local_path).convert("RGB"), local_path

    if not ALLOW_REMOTE_IMAGE_DOWNLOAD:
        raise FileNotFoundError(
            f"Image not found under {COCO_DIR}: {file_name}. "
            "Download COCO images on HPC or explicitly enable the Colab fallback."
        )

    coco_split = "val2014" if "val2014" in file_name else "train2014"
    image_url = f"https://images.cocodataset.org/{coco_split}/{file_name}"
    response = requests.get(image_url, verify=COCO_VERIFY_SSL, timeout=30)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB"), image_url

In [ ]:
from io import BytesIO

row = ds[0]
phrase = row["sentences"][0]["sent"]
file_name = row["file_name"]
bbox = [float(value) for value in row["bbox"]]

# The downloaded RefCOCOg examples report bbox as [x1, y1, x2, y2].
assert len(bbox) == 4
x1, y1, x2, y2 = bbox
assert x2 >= x1 and y2 >= y1

image, image_source = load_coco_image(file_name)

print("Phrase:", phrase)
print("Image source:", image_source)
print("Image size:", image.size)
print("Ground-truth box:", bbox)

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(image)
ax.add_patch(
    patches.Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        linewidth=3,
        edgecolor="red",
        facecolor="none",
    )
)
ax.set_title(phrase)
ax.axis("off")
fig.savefig(OUTPUT_DIR / "example.png", bbox_inches="tight")
plt.show()

In [ ]:
try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch is not installed in this notebook environment.")